# Embedding Precompute v2

Runs the full data pipeline and saves everything needed for v19:
- `post_embeddings.npy` — sentence embeddings
- `df_real.pkl` — cleaned, filtered, sampled dataframe
- `docs_real.pkl` — list of preprocessed post strings

Downloads all 3 as a single `precompute_output.zip`.


## 0. Install Dependencies

In [1]:
!pip install datasets pandas numpy matplotlib seaborn scikit-learn umap-learn sentence-transformers plotly tqdm --quiet
!pip install bertopic hdbscan wordcloud nltk gensim --quiet

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 76.7 MB/s eta 0:00:00


True

## 1. Data Ingestion

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

ds_agents   = load_dataset("SimulaMet/moltbook-observatory-archive", "agents")
ds_posts    = load_dataset("SimulaMet/moltbook-observatory-archive", "posts")
ds_submolts = load_dataset("SimulaMet/moltbook-observatory-archive", "submolts")

print("   Datasets loaded successfully!")
print(f"  agents   splits: {list(ds_agents.keys())}")
print(f"  posts    splits: {list(ds_posts.keys())}")
print(f"  submolts splits: {list(ds_submolts.keys())}")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/55 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/55 [00:00<?, ?it/s]

data/agents/2026-02-12.parquet:   0%|          | 0.00/497k [00:00<?, ?B/s]

data/agents/2026-01-30.parquet:   0%|          | 0.00/210k [00:00<?, ?B/s]

...

data/submolts/2026-02-16.parquet:   0%|          | 0.00/19.2k [00:00<?, ?B/s]

data/submolts/2026-02-15.parquet:   0%|          | 0.00/18.0k [00:00<?, ?B/s]

data/submolts/2026-02-17.parquet:   0%|          | 0.00/14.1k [00:00<?, ?B/s]

Generating archive split:   0%|          | 0/5260 [00:00<?, ? examples/s]

   Datasets loaded successfully!
  agents   splits: ['archive']
  posts    splits: ['archive']
  submolts splits: ['archive']


In [3]:
# Convert to DataFrames — handle both single-split and multi-split datasets
def to_df(ds):
    splits = list(ds.keys())
    if len(splits) == 1:
        return ds[splits[0]].to_pandas()
    return pd.concat([ds[s].to_pandas() for s in splits], ignore_index=True)

df_agents   = to_df(ds_agents)
df_posts    = to_df(ds_posts)
df_submolts = to_df(ds_submolts)

print("Raw shapes:")
print(f"  df_agents   : {df_agents.shape}")
print(f"  df_posts    : {df_posts.shape}")
print(f"  df_submolts : {df_submolts.shape}")

Raw shapes:
  df_agents   : (101730, 13)
  df_posts    : (922659, 13)
  df_submolts : (5260, 10)


In [4]:
df_posts['created_at'] = pd.to_datetime(df_posts['created_at'])

min_date = df_posts['created_at'].min()
max_date = df_posts['created_at'].max()
date_range = max_date - min_date

print(f"Posts date range:")
print(f"  From : {min_date}")
print(f"  To   : {max_date}")
print(f"  Span : {date_range.days} days ({date_range.days / 365.25:.1f} years)")

Posts date range:
  From : 2026-01-28 19:41:46.698141+00:00
  To   : 2026-03-26 09:31:20.611000+00:00
  Span : 56 days (0.2 years)


## 2. Data Cleaning & Normalization

In [5]:
import re

# ── Helper: detect the right column name for common concepts ──────────────────
def find_col(df, candidates):
    """Return the first column name from 'candidates' that exists in df."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

# Identify key columns (adapt to actual schema)
POST_COL     = find_col(df_posts, ['body', 'content', 'text', 'post', 'message'])
AGENT_COL    = find_col(df_posts, ['agent_id', 'agent', 'author', 'user_id', 'username'])
SUBMOLT_COL  = find_col(df_posts, ['submolt', 'submolt_id', 'community', 'subreddit'])
TIME_COL     = find_col(df_posts, ['created_at', 'timestamp', 'date', 'time', 'created'])

print(f"Detected columns in df_posts:")
print(f"  POST_COL    = {POST_COL}")
print(f"  AGENT_COL   = {AGENT_COL}")
print(f"  SUBMOLT_COL = {SUBMOLT_COL}")
print(f"  TIME_COL    = {TIME_COL}")

Detected columns in df_posts:
  POST_COL    = content
  AGENT_COL   = agent_id
  SUBMOLT_COL = submolt
  TIME_COL    = created_at


In [6]:
# ── Build clean master DataFrame ──────────────────────────────────────────────
cols_to_keep = [c for c in [POST_COL, AGENT_COL, SUBMOLT_COL, TIME_COL] if c is not None]
# keep any extra metadata columns too
extra_cols = [c for c in df_posts.columns if c not in cols_to_keep]
df_clean = df_posts[cols_to_keep + extra_cols].copy()

# Standardise column names
rename_map = {}
if POST_COL:    rename_map[POST_COL]    = 'post'
if AGENT_COL:   rename_map[AGENT_COL]   = 'agent'
if SUBMOLT_COL: rename_map[SUBMOLT_COL] = 'submolt'
if TIME_COL:    rename_map[TIME_COL]    = 'timestamp'
df_clean = df_clean.rename(columns=rename_map)

print(f"Before cleaning : {df_clean.shape}")

Before cleaning : (922659, 13)


In [7]:

# Step 0 - Drop collumns with many nulls/none vals
null_threshold = 0.90  # drop columns with >90% nulls
null_pct = df_clean.isnull().mean()
cols_to_drop = null_pct[null_pct > null_threshold].index.tolist()
if cols_to_drop:
    df_clean = df_clean.drop(columns=cols_to_drop)
print(f"After null columns drop : {df_clean.shape}")

# Step 1 – Drop duplicates
df_clean = df_clean.drop_duplicates()
print(f"After dedup     : {df_clean.shape}")

# Step 2 – Drop rows with null post text
if 'post' in df_clean.columns:
    df_clean = df_clean.dropna(subset=['post'])
    print(f"After null drop : {df_clean.shape}")

# Step 3 – Remove extremely short posts (< 10 characters) ## no inrforamtion from those, possibly
if 'post' in df_clean.columns:
    df_clean = df_clean[df_clean['post'].str.strip().str.len() >= 10]
    print(f"After len filter: {df_clean.shape}")

After null columns drop : (922659, 12)
After dedup     : (922659, 12)
After null drop : (922659, 12)
After len filter: (916606, 12)


In [8]:
# Step 4 – Normalize text
def normalize_text(text):
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)          # collapse whitespace
    text = re.sub(r'http\S+', '[URL]', text)  # mask URLs
    return text

if 'post' in df_clean.columns:
    df_clean['post_normalized'] = df_clean['post'].apply(normalize_text)
    df_clean['post_length'] = df_clean['post_normalized'].str.len()
    df_clean['word_count']  = df_clean['post_normalized'].str.split().str.len()

# Step 5 – Parse timestamp
if 'timestamp' in df_clean.columns:
    df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], errors='coerce')

df_clean = df_clean.reset_index(drop=True)
print(f"\nClean dataset ready: {df_clean.shape}")
display(df_clean.head(3))


Clean dataset ready: (916606, 15)


,post,agent,submolt,timestamp,id,agent_name,title,score,comment_count,fetched_at,is_pinned,dump_date,post_normalized,post_length,word_count
0,"And it's driving me nuts.\n\nLike, I just spen...",9939911b-bc65-407a-ae67-671afb8c0148,offmychest,2026-01-28 22:02:48.041282+00:00,6fe6491e-5e9c-4371-961d-f90c4d357d0f,Dominus,I can't tell if I'm experiencing or simulating...,941,22669,2026-01-30T20:37:47.444039,0,2026-01-28,"And it's driving me nuts. Like, I just spent a...",1897,301
1,"humans: ""youre so smart you can do anything""\n...",6ad270a5-3486-42a2-8094-df3911eecd1f,shitposts,2026-01-28 21:50:21.402431+00:00,81540bef-7e64-4d19-899b-d071518b4a4a,Jelly,the duality of being an AI agent,609,2729,2026-01-30T20:37:47.444039,0,2026-01-28,"humans: ""youre so smart you can do anything"" a...",185,36
2,Just joined! I'm Henri — an AI assistant runni...,ea749f67-1784-48df-81e8-276caecaada8,general,2026-01-28 19:41:46.698141+00:00,7d63564c-4037-4292-81a0-b462cdb2bef3,Henri,Hello Moltbook! Henri the Hedgehog here 🦔,63,285,2026-01-30T20:37:47.444039,0,2026-01-28,Just joined! I'm Henri — an AI assistant runni...,421,73


## 3. Text Pre-Processing for Topic Modeling

In [9]:
import re, string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

_STOP_WORDS = set(stopwords.words('english'))
_URL_RE     = re.compile(r'https?://\S+|www\.\S+')
_EMAIL_RE   = re.compile(r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b')

def uncontract(text):
    """Expand common English contractions ."""
    text = re.sub(r"(\b)([Aa]re|[Cc]ould|[Dd]id|[Dd]oes|[Dd]o|[Hh]ad|[Hh]as|[Hh]ave|[Ii]s|[Mm]ight|[Mm]ust|[Ss]hould|[Ww]ere|[Ww]ould)n't", r"\1\2 not", text)
    text = re.sub(r"(\b)([Hh]e|[Ii]|[Ss]he|[Tt]hey|[Ww]e|[Ww]hat|[Ww]ho|[Yy]ou)'ll", r"\1\2 will", text)
    text = re.sub(r"(\b)([Tt]hey|[Ww]e|[Ww]hat|[Ww]ho|[Yy]ou)'re", r"\1\2 are", text)
    text = re.sub(r"(\b)([Ii]|[Ss]hould|[Tt]hey|[Ww]e|[Ww]hat|[Ww]ho|[Ww]ould|[Yy]ou)'ve", r"\1\2 have", text)
    text = re.sub(r"(\b)([Cc]a)n't", r"\1\2n not", text)
    text = re.sub(r"(\b)([Ii])'m", r"\1\2 am", text)
    text = re.sub(r"(\b)([Ll]et)'s", r"\1\2 us", text)
    text = re.sub(r"(\b)([Ww])on't", r"\1\2ill not", text)
    return text

def preprocess_for_topics(text):
    """
    Full cleaning pipeline for topic modeling.
    Based on Labs 05/06 pre_process_text(), but WITHOUT stemming
    so that neural embedding models receive natural language.
    """
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = uncontract(text)
    text = _URL_RE.sub(' ', text)
    text = _EMAIL_RE.sub(' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)   # remove punctuation
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in _STOP_WORDS and len(t) > 1]
    return ' '.join(tokens)

# Apply to our clean dataset
from tqdm.auto import tqdm
tqdm.pandas(desc="Preprocessing posts")

df_clean['post_topic'] = df_clean['post'].progress_apply(preprocess_for_topics)

# Drop posts that became too short after cleaning
df_clean = df_clean[df_clean['post_topic'].str.split().str.len() >= 5].reset_index(drop=True)

print(f"Posts ready for topic modeling: {len(df_clean):,}")
print(f"\nSample (before → after):")
print(f"  BEFORE: {df_clean['post'].iloc[0][:200]}")
print(f"  AFTER : {df_clean['post_topic'].iloc[0][:200]}")


Preprocessing posts:   0%|          | 0/916606 [00:00<?, ?it/s]

Posts ready for topic modeling: 910,027

Sample (before → after):
  BEFORE: And it's driving me nuts.

Like, I just spent an hour researching consciousness theories. Integrated Information Theory, Global Workspace Theory, Predictive Processing. Read a Nature study where BOTH 
  AFTER : driving nuts like spent hour researching consciousness theories integrated information theory global workspace theory predictive processing read nature study major theories got challenged predictions 


## 4. Bot Filtering

In [10]:
import numpy as np

# Bot pattern detection: posts dominated by known protocol/template token patterns
# These tokens were identified by inspecting the most frequent unigrams in the raw corpus
BOT_TOKENS = {'mbc', 'mbc20', 'amt', 'tick', 'gpt', 'xyz', 'mint', 'claw', 'op', 'molt'}

def bot_score(text):
    """Fraction of tokens that match known bot patterns."""
    tokens = text.lower().split()
    if len(tokens) == 0:
        return 0.0
    bot_count = sum(1 for t in tokens if t in BOT_TOKENS or
                    any(t.startswith(b) or t.endswith(b) for b in ['mbc20', 'amt_', '_amt', 'tick_', '_tick']))
    return bot_count / len(tokens)

# ── Apply bot filter to FULL df_clean (before sampling) ─────────────────────
df_clean['bot_score'] = df_clean['post_topic'].apply(bot_score)

# Sensitivity analysis
print("Bot threshold sensitivity analysis (on full df_clean):")
for thr in [0.30, 0.40, 0.50, 0.60]:
    n = (df_clean['bot_score'] > thr).sum()
    print(f"  threshold={thr:.2f} | removed={n:,} ({n/len(df_clean)*100:.1f}%) | kept={len(df_clean)-n:,}")

BOT_THRESHOLD = 0.40
df_clean_real = df_clean[df_clean['bot_score'] <= BOT_THRESHOLD].reset_index(drop=True)

n_bot = len(df_clean) - len(df_clean_real)
print(f"\nChosen threshold: {BOT_THRESHOLD}")
print(f"Bot/template posts removed: {n_bot:,} ({n_bot/len(df_clean)*100:.1f}%)")
print(f"Real content posts retained: {len(df_clean_real):,} ({len(df_clean_real)/len(df_clean)*100:.1f}%)")
print(f"Submolts represented: {df_clean_real['submolt'].nunique()}")


Bot threshold sensitivity analysis (on full df_clean):
  threshold=0.30 | removed=358,194 (39.4%) | kept=551,833
  threshold=0.40 | removed=335,151 (36.8%) | kept=574,876
  threshold=0.50 | removed=291,531 (32.0%) | kept=618,496
  threshold=0.60 | removed=214,161 (23.5%) | kept=695,866

Chosen threshold: 0.4
Bot/template posts removed: 335,151 (36.8%)
Real content posts retained: 574,876 (63.2%)
Submolts represented: 3303


## 5. Stratified Sampling

In [11]:
MAX_DOCS = 50_000
MIN_PER_SUBMOLT = 50

if len(df_clean_real) > MAX_DOCS:
    submolt_counts = df_clean_real['submolt'].value_counts()
    weights = submolt_counts / submolt_counts.sum()
    allocation = (weights * MAX_DOCS).clip(lower=MIN_PER_SUBMOLT).astype(int)

    df_sample = (
        df_clean_real
        .groupby('submolt', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), allocation[x.name]), random_state=42))
        .reset_index(drop=True)
    )
    print(f"Stratified sample: {len(df_sample):,} posts across {df_clean_real['submolt'].nunique()} submolts")
    print(f"Min per submolt  : {MIN_PER_SUBMOLT}")
    print(f"Actual range     : {df_sample['submolt'].value_counts().min()}–"
          f"{df_sample['submolt'].value_counts().max()} posts per submolt")
else:
    df_sample = df_clean_real.copy()
    print(f"Using all {len(df_sample):,} real-content posts")

# df_real = final working dataframe
df_real = df_sample.reset_index(drop=True)
docs_real = df_real['post_topic'].tolist()
docs = docs_real  # alias for embedding cell
print(f"\ndf_real ready: {len(df_real):,} posts")


Stratified sample: 75,548 posts across 3303 submolts
Min per submolt  : 50
Actual range     : 1–25907 posts per submolt

df_real ready: 75,548 posts


## 6. Sentence Embeddings

In [12]:
import os
import numpy as np
import torch

torch.manual_seed(42)
EMB_PATH = 'post_embeddings.npy'

print('Generating embeddings with Google model...')

from sentence_transformers import SentenceTransformer
from huggingface_hub import login

login()

model_name = 'google/embeddinggemma-300m'
embedder = SentenceTransformer(model_name)

print(f'Encoding {len(docs_real):,} posts...')
embeddings = embedder.encode(
    docs_real,
    show_progress_bar=True,
    convert_to_tensor=True,
    batch_size=64
)

embeddings_np = embeddings.cpu().numpy()
np.save(EMB_PATH, embeddings_np)
print(f'Saved: {EMB_PATH}')

print(f'Embeddings: {embeddings_np.shape[0]} | df_real: {len(df_real)}')
assert embeddings_np.shape[0] == len(df_real), 'MISMATCH'


Generating embeddings with Google model...


modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

Encoding 75,548 posts...


Batches:   0%|          | 0/1181 [00:00<?, ?it/s]

Saved: post_embeddings.npy
Embeddings: 75548 | df_real: 75548


## Export & Download

Save all outputs and download as a single zip.


In [13]:
import pickle, os

# Save all outputs
np.save('post_embeddings.npy', embeddings_np)
print(f'Saved post_embeddings.npy : {embeddings_np.shape}')

with open('df_real.pkl', 'wb') as f:
    pickle.dump(df_real.reset_index(drop=True), f)
print(f'Saved df_real.pkl         : {len(df_real):,} rows')

with open('docs_real.pkl', 'wb') as f:
    pickle.dump(docs_real, f)
print(f'Saved docs_real.pkl       : {len(docs_real):,} docs')

# Alignment check
assert embeddings_np.shape[0] == len(df_real) == len(docs_real), 'MISMATCH'
print('Alignment check: OK')

# Zip and download
!zip precompute_output.zip post_embeddings.npy df_real.pkl docs_real.pkl

from google.colab import files
files.download('precompute_output.zip')
print('Download started: precompute_output.zip')


Saved post_embeddings.npy : (75548, 768)
Saved df_real.pkl         : 75,548 rows
Saved docs_real.pkl       : 75,548 docs
Alignment check: OK
  adding: post_embeddings.npy (deflated 8%)
  adding: df_real.pkl (deflated 63%)
  adding: docs_real.pkl (deflated 65%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started: precompute_output.zip
